# Lab 3 — QSCI as an HPC workflow: QDMI quantum node → Slurm → AMD GPU

**Workshop block 5 · 30 min (optional)** · builds on *"Using Quantum-Selected Configuration Interaction with AMD GPUs"*
(IQM, N. Steinmann & D. Bulmash) and on the QDMI integration path of arXiv:2604.19869.

The original notebook connected one Python process directly to IQM Resonance (`IQMProvider`) and ran the
post-processing on an AMD GPU droplet. Here we make it look like a **production HPC job**:

| Stage | Where it runs | How it is reached |
|---|---|---|
| Classical preparation: molecule, RHF, UCCSD ansatz | login node / this notebook | Python |
| **Quantum stage**: VQE + bitstring sampling | quantum access node (`quantum` partition) | **QDMI** → DDSIM *or* IQM Resonance |
| **Classical stage**: filter, subspace Hamiltonian, `eigvalsh` | GPU node (`gpu`/`compute` partition) | Slurm `--dependency=afterok`, JAX on ROCm |

The algorithm code is identical in every mode. Only two *backend spec strings* and the launch method change:

```
"statevector"                 exact, local, no QDMI          (debugging)
"qdmi:mqt.ddsim.default"      QDMI -> local DDSIM simulator  (classical node stands in for the QPU)
"qdmi:<iqm device id>"        QDMI -> IQM Resonance          (real or :mock quantum node)
```

In [ ]:
from IPython.display import Image, display
display(Image(filename="qsci_workflow.png", width=650))

### What this tutorial does, in plain words
You want to do linear algebra on a very large matrix, in principle too large to work with classically.

1. **Input the matrix.** Here it is the **Hamiltonian** of a molecule, which tells you the energy of the molecule as a
   function of the state of its electrons.
2. **Quantum part: shrink the matrix.** The QPU samples the electron configurations that matter, and we keep only those.
   This shrinks the matrix in a way that shouldn't change the output we are looking for.
3. **Classical (GPU) part: do the linear algebra** on the smaller matrix.
4. **Output:** the approximate **ground-state energy**, the lowest possible energy of the electrons in the molecule.

We run it on a simple molecule, **H₂**, so the answer can be checked against an exact calculation, but the same code
runs on more complicated molecules (section 8).

### The idea in two lines
The quantum computer samples the electron configurations $|\Phi_i\rangle$ that matter most from an exponentially
large space; the classical machine diagonalises $H^{\mathcal S}_{ij}=\langle\Phi_i|\hat H|\Phi_j\rangle$ on that
subspace $\mathcal S$: $\;E_0^{\rm QSCI}=\lambda_{\min}(H^{\mathcal S})\ge E_0$ (variational upper bound).
Building $H^{\mathcal S}$ costs $O(|\mathcal S|^2)$ matrix elements and diagonalising it $O(|\mathcal S|^3)$.
That is the part an AMD GPU is good at.

### What changed compared with the original notebook
* Quantum access goes through **QDMI** (`mqt.core.plugins.qiskit.QDMIBackend`), not a vendor provider, so the same code
  targets DDSIM, Resonance or an on-prem Radiance.
* The missing helpers (`estimate_*`, `sample_*`, `_full_mbh_canonical_from_qsci`) are implemented in `qsci_lib.py`
  (Slater–Condon rules, checked against brute-force second quantisation).
* VQE uses **COBYLA** (gradient-free, tolerant of shot noise) instead of L-BFGS-B with finite differences.
* The exact reference is PySCF **FCI** (avoids the `qiskit-algorithms 0.2.2` SciPy bug).
* Every stage reads/writes JSON in a shared run directory, so it can be a separate Slurm job.

## 0 · Environment

```bash
uv pip install -r requirements.txt               # CPU everywhere
# AMD GPU node / droplet (ROCm 6.x): install JAX's ROCm build instead of plain jax, e.g.
#   pip install "jax[rocm]"      or use AMD's ROCm JAX container  (see README, section "AMD GPU")
```

In [ ]:
# %pip install -q -r requirements.txt

In [ ]:
import json, os, time, datetime
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

import qsci_lib as Q
print("Slater-Condon self-test E_elec(H2, R=1.4 bohr) =", round(Q._selftest_h2(), 4), "Ha (expected -1.8516)")
print("Linear algebra devices:", Q.jax_device_report())
print("Slurm on this machine:", Q.slurm_available())

## 1 · Configure the run

Start with everything local. Then flip **one string at a time**: first `sample_backend` to an IQM `:mock` device,
then to real hardware, then `RUN_MODE = "slurm"`.

In [ ]:
RUN_MODE = "inline"                  # "inline" (this kernel) | "slurm" (two sbatch jobs)

cfg = Q.QSCIConfig(
    atom="H 0 0 0; H 0 0 1",         # try "Li 0 0 0; H 0 0 1.6" or an H4 chain later
    basis="sto-3g",
    vqe_backend="statevector",       # cheapest: optimise classically ...
    sample_backend="qdmi:mqt.ddsim.default",   # ... sample through QDMI (local DDSIM for now)
    vqe_shots=1000, vqe_maxiter=60,
    sample_shots=2000, cutoff=10, seed=7,
)

RUNS = Path(os.environ.get("QSCI_RUNS", "runs"))   # on a cluster: a shared filesystem path, e.g. /jobs/qsci
run_dir = RUNS / datetime.datetime.now().strftime("qsci-%Y%m%d-%H%M%S")
cfg.save(run_dir); print("run dir:", run_dir.resolve())

## 2 · Classical preparation (login node)

PySCF computes the RHF reference and integrals; Qiskit Nature maps the fermionic Hamiltonian to qubits with
Jordan–Wigner (spin orbital *p* → qubit *p*, α block first) and builds the HF-initialised UCCSD ansatz.

In [ ]:
problem = Q.build_problem(cfg.atom, cfg.basis)
ansatz, observable = Q.build_ansatz_and_observable(problem)
print(f"spatial orbitals={problem.num_spatial_orbitals}  particles={problem.num_particles}")
print(f"qubits={ansatz.num_qubits}  UCCSD parameters={ansatz.num_parameters}  Pauli terms={len(observable)}")
print(f"qubit-wise commuting measurement groups = {len(observable.group_commuting(qubit_wise=True))}"
      "  (= circuits per energy evaluation on hardware)")

What does the ansatz cost on IQM hardware? Ask the device through QDMI and transpile to its native `prx`/`cz` target.
(Uses a QDMI session but submits nothing.)

In [ ]:
from qiskit import transpile
from mqt.core.qdmi import builtin_driver
iqm_ids = [d for d in builtin_driver.registered_device_ids() if d.lower().startswith("iqm")]
print("IQM QDMI devices:", iqm_ids)
if iqm_ids and (os.environ.get("RESONANCE_API_KEY") or os.environ.get("IQM_TOKENS_FILE")):
    b = Q.open_qdmi_backend(os.environ.get("IQM_DEVICE_ID") or iqm_ids[0])
    t = transpile(ansatz.assign_parameters(np.zeros(ansatz.num_parameters)), backend=b, optimization_level=3)
    print(f"{b.name}: depth={t.depth()}  ops={dict(t.count_ops())}")
else:
    print("set RESONANCE_API_KEY (see Lab 1) to inspect the IQM-native circuit")

## 3 · Quantum stage: VQE, then sampling

`make_energy_fn` returns either an exact statevector energy or a `QDMIEnergy`, which groups Paulis into
qubit-wise-commuting sets, transpiles one measurement circuit per group **once** to the device target, and re-binds
parameters every iteration. Each evaluation = one batch of QDMI jobs.

In [ ]:
energy_sv = Q.make_energy_fn("statevector", ansatz, observable, cfg)
vqe_sv = Q.run_vqe(energy_sv, ansatz.num_parameters, cfg.vqe_maxiter)

energy_qdmi = Q.make_energy_fn("qdmi:mqt.ddsim.default", ansatz, observable, cfg)
vqe_qdmi = Q.run_vqe(energy_qdmi, ansatz.num_parameters, 30)

plt.plot(np.array(vqe_sv["history"]) + problem.nuclear_repulsion_energy, label="statevector (exact)")
plt.plot(np.array(vqe_qdmi["history"]) + problem.nuclear_repulsion_energy, ".-", alpha=.7,
         label=f"QDMI/DDSIM, {cfg.vqe_shots} shots ({energy_qdmi.circuits_submitted} circuits)")
plt.xlabel("energy evaluation"); plt.ylabel("E total (Ha)"); plt.legend(); plt.title("VQE convergence"); plt.show()
print(f"statevector: {vqe_sv['seconds']:.1f}s | QDMI/DDSIM: {vqe_qdmi['seconds']:.1f}s")

In [ ]:
counts = Q.sample_counts(cfg.sample_backend, ansatz, vqe_sv["theta"], cfg.sample_shots, cfg)
top = sorted(counts.items(), key=lambda kv: -kv[1])[:12]
plt.bar([k for k, _ in top], [v for _, v in top]); plt.xticks(rotation=60)
plt.title(f"sampled configurations via {cfg.sample_backend}"); plt.ylabel("counts"); plt.show()
sel = Q.select_configurations(counts, ansatz.num_qubits, problem.num_particles, cfg.cutoff)
print("physical (N_alpha, N_beta) configurations kept:", [format(s, f"0{ansatz.num_qubits}b") for s in sel])

## 4 · Classical stage: subspace Hamiltonian + JAX `eigvalsh`

On an AMD GPU node with ROCm JAX, `jax.devices()` shows `rocm` devices and the dense diagonalisation runs there
automatically. Without JAX the same code falls back to NumPy.

In [ ]:
res = Q.postprocess_gpu(cfg.atom, cfg.basis, counts, cfg.cutoff, cfg.tol)
e_fci = Q.fci_reference(cfg.atom, cfg.basis)
print(json.dumps({k: res[k] for k in ["e_total", "e_hf", "n_selected", "nnz", "linear_algebra", "devices"]}, indent=1))
print(f"E_QSCI = {res['e_total']:.8f} | E_FCI = {e_fci:.8f} | error = {1000*(res['e_total']-e_fci):.4f} mHa")

## 5 · The same workflow as two Slurm jobs

`submit_pipeline` submits `slurm/quantum_stage.sbatch` to the **quantum** partition with a QDMI device **licence**,
then `slurm/classical_stage.sbatch` to the GPU/compute partition with `--dependency=afterok`. Both stages share the
run directory. This is exactly the "chain quantum and classical stages" pattern of the MQT playground and IQM's HPC
Integration Service.

Where to run this cell:
* **MQT Docker playground** (Lab 1, step 5): JupyterLab or a Python shell on the `login` container; code under `/jobs/qsci`; `QSCI_RUNS=/jobs/qsci/runs`.
* **AMD droplet**: run `slurm/single_node_slurm_droplet.sh` once → partitions `quantum` + `gpu` on one node.
* **Your cluster**: set partitions/account below (e.g. LUMI-G: `classical_partition="small-g"`, add `--account=project_xxx`).

In [ ]:
SLURM = dict(
    device_license=None,             # None -> derived from cfg.sample_backend, e.g. "mqt.ddsim.default"
    quantum_partition="quantum",
    classical_partition="compute",   # "gpu" on the droplet script, "small-g"/"standard-g" on LUMI
    gpus=0,                          # 1 on an AMD GPU partition
    extra_quantum=(), extra_classical=(),   # e.g. ("--account=project_465000000",)
)

if RUN_MODE == "slurm":
    assert Q.slurm_available(), "no sbatch here: run this on a login node"
    lic = SLURM["device_license"] or cfg.sample_backend.split(":", 1)[1]
    qid, cid = Q.submit_pipeline(run_dir, "slurm/quantum_stage.sbatch", "slurm/classical_stage.sbatch",
                                 device_license=lic, **{k: v for k, v in SLURM.items() if k != "device_license"})
    print(f"quantum job {qid} -> classical job {cid}")
    Q.wait_for([qid, cid])
    result = json.loads((run_dir / "result.json").read_text())
else:
    print("RUN_MODE='inline': running both stages in this kernel, same code as the batch jobs")
    result = Q.run_inline(cfg, run_dir)
print(f"\nE_QSCI = {result['e_total']:.8f} Ha  (FCI {result['e_fci']:.8f}, error {result['error_mHa']:.3f} mHa)")
print(f"classical stage ran on {result['host']} using {result['linear_algebra']} [{result['devices']}]")

## 6 · Point the quantum stage at IQM Resonance

1. Export your token (Lab 1). In Slurm the submission environment is exported to the job by default.
2. Use a **`:mock`** device first (free), then the real QPU.
3. Keep the cheap split: optimise on `statevector`, **sample** on the QPU. Then, if credits allow, move VQE to the QPU
   too (`vqe_backend="qdmi:<id>"`, `vqe_maxiter≈15`, `vqe_shots≈100`).

> **Credits.** VQE on hardware costs ≈ `iterations × groups × shots`. For H₂ (≈5 groups) the original notebook's
> 100-shot VQE + 100-shot sampling used about 45 credits on IQM Garnet. Sampling-only is a small fraction of that.

In [ ]:
IQM_DEVICE = os.environ.get("IQM_DEVICE_ID") or next((d for d in iqm_ids if "mock" in d), None)
if IQM_DEVICE:
    hw = Q.QSCIConfig(**{**cfg.__dict__, "sample_backend": f"qdmi:{IQM_DEVICE}", "sample_shots": 500})
    hw_dir = RUNS / (run_dir.name + "-iqm")
    hw_result = Q.run_inline(hw, hw_dir) if RUN_MODE == "inline" else None
    if RUN_MODE == "slurm":
        hw.save(hw_dir)
        ids = Q.submit_pipeline(hw_dir, "slurm/quantum_stage.sbatch", "slurm/classical_stage.sbatch",
                                device_license=IQM_DEVICE, **{k: v for k, v in SLURM.items() if k != "device_license"})
        Q.wait_for(list(ids)); hw_result = json.loads((hw_dir / "result.json").read_text())
    print(f"{IQM_DEVICE}: E_QSCI = {hw_result['e_total']:.6f} Ha, error {hw_result['error_mHa']:.2f} mHa,"
          f" kept {hw_result['n_selected']} configs {hw_result['selected_states']}")
else:
    print("No IQM device configured: set IQM_DEVICE_ID or install iqm-qdmi and export your token.")

Why is the hardware result often *still* close to FCI for H₂? Noise spreads counts over wrong-particle-number
bitstrings, and the **symmetry filter** discards them. As long as the important determinants are sampled at all,
the classical diagonalisation recovers the energy. This robustness is one reason QSCI-type methods suit today's
hardware; the price is moved to the classical side, which grows with $|\mathcal S|$.

## 7 · Where the GPU earns its keep

H₂/STO-3G has only 4 determinants with the right symmetry, so the GPU is idle. The cell below times the classical
kernel QSCI hits at scale (dense Hermitian `eigvalsh`) on whatever JAX sees, CPU or AMD GPU, for growing $|\mathcal S|$.

In [ ]:
try:
    import jax, jax.numpy as jnp
    jax.config.update("jax_enable_x64", True)
    sizes, times = [256, 512, 1024, 2048, 4096], []
    key = jax.random.PRNGKey(0)
    f = jax.jit(jnp.linalg.eigvalsh)
    for n in sizes:
        A = jax.random.normal(key, (n, n)); A = (A + A.T) / 2
        f(A).block_until_ready()                      # compile
        t0 = time.perf_counter(); f(A).block_until_ready(); times.append(time.perf_counter() - t0)
        print(f"|S| = {n:5d}: eigvalsh {times[-1]*1e3:8.1f} ms on {jax.devices()[0].platform}")
    plt.loglog(sizes, times, "o-"); plt.xlabel("|S|"); plt.ylabel("s"); plt.title("dense eigvalsh, ~|S|^3"); plt.show()
except ImportError:
    print("install jax (ROCm build on AMD GPUs) to run the benchmark")

## 8 · Take it further

* **Bigger molecules:** `atom="Li 0 0 0; H 0 0 1.6"` (12 qubits) or `"H 0 0 0; H 0 0 0.9; H 0 0 1.8; H 0 0 2.7"`.
  Raise `cutoff` and `sample_shots`; watch `n_selected`, `nnz` and the error vs FCI.
* **Beyond dense:** for $|\mathcal S|\gtrsim 10^4$ keep $H^{\mathcal S}$ sparse (`jax.experimental.sparse.BCSR`) and use a
  Lanczos/Davidson solver; vectorise the Slater–Condon loop with `jax.vmap` over determinant pairs.
* **Tighter loop:** request a heterogeneous Slurm allocation (`sbatch ... : ...`) with a QAN and a GPU node, so iterative
  variants (sample → diagonalise → update ansatz) don't re-queue every step.

### Methodology review
The QSCI energy is a variational upper bound. The gap to FCI depends on how much of the true wavefunction lies
outside $\mathcal S$: a weak ansatz, too few shots or hardware noise widen it, while a larger `cutoff` narrows it
at $O(|\mathcal S|^3)$ classical cost.